## 🎯 Learning Objectives
* Understand the importance of defining structured outputs for AI agent tasks.
* Learn how to use Pydantic models to specify expected output schemas for CrewAI tasks.
* Implement a CrewAI task that enforces a specific JSON output structure.
* Interpret and utilize structured outputs from AI agents for downstream processing.


## Defining Tasks with Expected Output Schemas in CrewAI

In the world of AI agents, especially when building complex, multi-agent systems like those powered by CrewAI, the clarity and consistency of information flow are paramount. Imagine you're building an automated pipeline where one agent researches a topic, another summarizes it, and a third drafts a report. If the research agent provides its findings in an inconsistent, free-form text, the summarizer agent will struggle to reliably extract the necessary information, leading to errors, hallucinations, and a broken pipeline.

This is where **defining tasks with expected output schemas** becomes a game-changer. It's like providing a strict contract or a blueprint for the agent's output. Instead of asking an agent to "summarize this article," you're asking it to "summarize this article and return the summary in a JSON format with a `title` field, a list of `key_points`, and a `sentiment` analysis, all as strings."

### Why is this crucial?

1.  **Reliability and Consistency**: Agents are prompted to adhere to a specific structure, significantly reducing the chances of free-form, unparseable, or inconsistent outputs.
2.  **Downstream Processing**: Structured outputs (like JSON) are inherently machine-readable. This makes it trivial for subsequent agents, databases, APIs, or other software components to consume and process the information without complex parsing logic.
3.  **Reduced Hallucinations**: By guiding the LLM towards a specific output format, you implicitly constrain its generation space, often leading to more focused and less hallucinatory responses.
4.  **Easier Debugging**: When an agent's output doesn't match the schema, it's immediately apparent, making debugging and refinement much simpler.
5.  **Data Validation**: Leveraging tools like Pydantic (which CrewAI integrates with) allows for automatic validation of the generated output against the defined schema, catching errors early.

### The Analogy: A Mold for Concrete

Think of an expected output schema as a **mold for pouring concrete**. If you want a specific shape (a brick, a statue, a foundation), you use a mold. Without it, the concrete (the agent's output) would just spread out formlessly. The mold ensures the concrete takes the desired, predictable shape, making it useful for its intended purpose. Similarly, an output schema provides the 'mold' for your agent's textual output, ensuring it's structured and ready for its next step.

### How CrewAI Leverages Pydantic

CrewAI integrates seamlessly with **Pydantic**, a powerful Python library for data validation and settings management. You define your desired output structure using a Pydantic `BaseModel`, and then pass this model to the `output_pydantic` parameter of your `Task` definition. CrewAI then automatically injects instructions into the agent's prompt, guiding the underlying Large Language Model (LLM) to produce output that conforms to this schema.

Let's see this in action.


In [ ]:
# Ensure you have the necessary libraries installed
# pip install crewai pydantic 'langchain-openai>=0.1.1'

import os
from crewai import Agent, Task, Crew, Process
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import List

# --- 1. Set up your API Key (replace with your actual key or environment variable) ---
# It's recommended to set this as an environment variable for security.
# For demonstration, we'll set it directly. In production, use `os.getenv('OPENAI_API_KEY')`
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# Initialize the LLM (using OpenAI's GPT-4o for robust schema adherence)
llm = ChatOpenAI(model="gpt-4o", temperature=0.2)

# --- 2. Define the Pydantic Model for the Expected Output ---
# This model specifies the exact structure we expect from our agent's output.
class ResearchSummary(BaseModel):
    title: str = Field(description="The concise title of the research summary.")
    key_points: List[str] = Field(description="A list of the most important findings or takeaways.")
    sentiment: str = Field(description="Overall sentiment of the research (e.g., 'Positive', 'Negative', 'Neutral', 'Mixed').")
    confidence_score: float = Field(description="A score from 0.0 to 1.0 indicating the agent's confidence in the summary.")

# --- 3. Define the Agent ---
# This agent will be responsible for performing the research and summarizing it.
researcher_agent = Agent(
    role='Senior Research Analyst',
    goal='Conduct in-depth research on a given topic and provide structured insights.',
    backstory='An expert in quickly sifting through vast amounts of information to extract core facts and trends.',
    verbose=True,
    allow_delegation=False,
    llm=llm
)

# --- 4. Define the Task with the Pydantic Output Schema ---
# Notice the `output_pydantic=ResearchSummary` parameter.
research_task = Task(
    description=(
        "Research the latest advancements in quantum computing for drug discovery. "
        "Focus on breakthroughs, challenges, and potential future impacts. "
        "Ensure the summary is concise and highlights the most critical aspects."
    ),
    expected_output="A structured JSON object conforming to the ResearchSummary Pydantic model.",
    agent=researcher_agent,
    output_pydantic=ResearchSummary, # <--- This is where we link the schema
    async_execution=False
)

# --- 5. Create and Run the Crew ---
# For this example, we have a single agent and a single task.
crew = Crew(
    agents=[researcher_agent],
    tasks=[research_task],
    process=Process.sequential, # Tasks are executed one after another
    verbose=2 # Shows more detailed logs
)

print("\n### Starting the Crew with Structured Output Task ###")
result = crew.kickoff()

print("\n### Crew Execution Finished ###")

# --- 6. Access and Validate the Structured Output ---
# The result will be an instance of our Pydantic model, not just a string.
if isinstance(result, ResearchSummary):
    print("\nOutput is a valid ResearchSummary object!")
    print(f"Title: {result.title}")
    print("Key Points:")
    for point in result.key_points:
        print(f"  - {point}")
    print(f"Sentiment: {result.sentiment}")
    print(f"Confidence Score: {result.confidence_score}")
    
    # You can also convert it back to a dictionary or JSON string if needed
    print("\nAs Dictionary:")
    print(result.model_dump())
    print("\nAs JSON String:")
    print(result.model_dump_json(indent=2))
else:
    print("\nOutput is NOT a ResearchSummary object. Something went wrong.")
    print(f"Raw output: {result}")


### Interpreting the Code Output and Practical Implications

When you run the code, you'll observe the CrewAI agent's thought process (`verbose=2`) as it attempts to fulfill the task. Crucially, the final output (`result`) will not be a raw string of text. Instead, it will be an instance of our `ResearchSummary` Pydantic model. This means:

*   **Direct Object Access**: You can access specific fields like `result.title`, `result.key_points`, and `result.sentiment` directly, just like any other Python object. This eliminates the need for manual string parsing (e.g., using regular expressions or custom parsers).
*   **Automatic Validation**: Pydantic automatically validates the data types and structure. If the LLM somehow fails to produce a valid `float` for `confidence_score` or a `list` for `key_points`, Pydantic will raise a validation error, immediately alerting you to an issue. This is a powerful debugging and quality control mechanism.
*   **Serialization**: The `model_dump()` method converts the Pydantic object into a standard Python dictionary, and `model_dump_json()` converts it into a JSON string. This makes it incredibly easy to pass this structured data to databases, APIs, or other services that expect JSON.

### Performance Trade-offs

While incredibly beneficial, enforcing output schemas does come with minor trade-offs:

*   **Increased Token Usage**: The Pydantic schema definition is injected into the LLM's prompt as instructions. This adds to the total token count for each request, potentially increasing API costs, especially for complex schemas or high-volume tasks.
*   **Slightly Longer Latency**: The LLM might take a fraction longer to generate output that strictly adheres to a complex schema, as it has an additional constraint to satisfy during generation.
*   **LLM Capability**: The effectiveness of schema adherence depends on the underlying LLM's instruction following capabilities. More advanced models (like GPT-4o, Claude 3 Opus) are generally better at this than smaller, less capable models.

### Typical Use Cases

Defining output schemas is invaluable in many agentic workflows:

*   **Data Extraction**: Extracting specific entities (names, dates, amounts) from unstructured text into a structured format.
*   **API Call Generation**: Generating parameters for API calls (e.g., a `search_query` string, `filters: List[str]`, `max_results: int`).
*   **Content Generation**: Creating structured content like blog post outlines (title, sections, keywords), product descriptions (name, features, benefits), or social media posts (text, hashtags, image_prompt).
*   **Database Interactions**: Preparing data to be inserted into a database, ensuring it matches the table schema.
*   **Inter-Agent Communication**: Ensuring that the output of one agent is perfectly formatted for consumption by another agent in a multi-step crew.

By mastering structured outputs, you elevate your AI agents from mere text generators to reliable, data-producing components within sophisticated automated systems.


### Resources

*   **CrewAI Documentation - Tasks**: [https://docs.crewai.com/core-concepts/Tasks/](https://docs.crewai.com/core-concepts/Tasks/)
*   **Pydantic Documentation**: [https://docs.pydantic.dev/latest/](https://docs.pydantic.dev/latest/)
*   **LangChain Output Parsers (CrewAI leverages similar concepts)**: [https://python.langchain.com/docs/modules/model_io/output_parsers/](https://python.langchain.com/docs/modules/model_io/output_parsers/)
*   **OpenAI API Documentation (for model capabilities)**: [https://platform.openai.com/docs/models](https://platform.openai.com/docs/models)
